# Pipeline 1 — Vanilla RAG (BM25 Sparse Retrieval)

**Method:** Classic BM25 keyword matching → top-k sentences → LLM answer generation.

```
Question ──→ BM25 Retrieve (top-k) ──→ LLM Generate Answer
```

**Features:**
- N-key Groq API carousel (all available keys) with rate-limit tracking and auto-rotation
- Parallel sample processing via ThreadPoolExecutor (14 workers)
- Safety buffers below Groq free-tier limits (25/30 RPM, 900/1000 RPD per key)

## Step 1 — Install Dependencies

In [ ]:
!pip install -q datasets numpy rank-bm25 groq tqdm ragas langchain-groq langchain-core langchain-community langchain-google-vertexai

## Step 1b — Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- Path to your project folder on Google Drive ---
BASE_DIR     = '/content/drive/MyDrive/HotPotQA-Coding-Trials'
API_KEYS_CSV = f'{BASE_DIR}/api_keys.csv'
RESULTS_DIR  = f'{BASE_DIR}/Results/BestEmbeddingComparison/JSONOutputs/Qdrant'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Drive mounted. Project dir: {BASE_DIR}")

## Step 2 — Imports and Configuration

In [ ]:
import os, json, re, time, random, collections, string, csv, threading
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from groq import Groq
from datetime import datetime

# --- Configuration ---
GROQ_MODEL      = "llama-3.3-70b-versatile"
TOP_K           = 4
N_SAMPLES     = 100
SEED            = 42
MAX_WORKERS   = None  # auto-set to len(key_manager.keys) after Step 3
PIPELINE_NAME = "VanillaRAG_e5-base-v2_Qdrant"
# API_KEYS_CSV and RESULTS_DIR are set in the Drive mount cell above

print(f"Pipeline : {PIPELINE_NAME}")
print(f"Model    : {GROQ_MODEL}")
print(f"Workers  : auto (set after API keys load)")
print(f"Top-K    : {TOP_K}")
print(f"Samples  : {N_SAMPLES}")

## Step 3 — API Key Manager (14-Key Carousel)

Loads all keys from `api_keys.csv` and automatically rotates when rate limits are approached.

| Limit | Groq Free Tier | Buffer (safety margin) |
|-------|---------------|------------------------|
| RPM   | 30            | **25**                 |
| RPD   | 1,000         | **900**                |
| TPM   | 12,000        | **10,000**             |
| TPD   | 100,000       | **90,000**             |

In [ ]:
class APIKeyManager:
    """Thread-safe API key carousel with per-key rate-limit tracking."""

    BUFFERS = {'rpm': 25, 'rpd': 900, 'tpm': 10_000, 'tpd': 90_000}

    def __init__(self, csv_path):
        self.keys = self._load_keys(csv_path)
        self._lock = threading.Lock()
        self._idx = 0
        self._usage = {
            k: {'min_req': [], 'day_req': [], 'min_tok': [], 'day_tok': []}
            for k in self.keys
        }
        print(f"Loaded {len(self.keys)} API keys from {csv_path}")

    # ---- key loading ----
    @staticmethod
    def _load_keys(csv_path):
        keys = []
        with open(csv_path, 'r') as f:
            for row in csv.DictReader(f):
                k = row.get('API_KEY', '').strip()
                if k:
                    keys.append(k)
        if not keys:
            raise ValueError(f"No keys in {csv_path}")
        return keys

    # ---- sliding-window cleanup ----
    def _clean(self, key):
        now = time.time()
        u = self._usage[key]
        u['min_req'] = [t for t in u['min_req'] if now - t < 60]
        u['day_req'] = [t for t in u['day_req'] if now - t < 86400]
        u['min_tok'] = [(t, n) for t, n in u['min_tok'] if now - t < 60]
        u['day_tok'] = [(t, n) for t, n in u['day_tok'] if now - t < 86400]

    def _is_available(self, key):
        self._clean(key)
        u = self._usage[key]
        return (
            len(u['min_req']) < self.BUFFERS['rpm']
            and len(u['day_req']) < self.BUFFERS['rpd']
            and sum(n for _, n in u['min_tok']) < self.BUFFERS['tpm']
            and sum(n for _, n in u['day_tok']) < self.BUFFERS['tpd']
        )

    # ---- carousel ----
    def get_key(self):
        with self._lock:
            for _ in range(len(self.keys)):
                key = self.keys[self._idx]
                self._idx = (self._idx + 1) % len(self.keys)
                if self._is_available(key):
                    return key
            return self._wait_and_get()

    def _wait_and_get(self):
        min_wait = 60
        for key in self.keys:
            reqs = self._usage[key]['min_req']
            if reqs:
                wait = 60 - (time.time() - min(reqs))
                min_wait = min(min_wait, max(0, wait))
        print(f"  ⏳ All keys busy — waiting {min_wait:.1f}s for RPM reset...")
        time.sleep(min_wait + 1)
        for _ in range(len(self.keys)):
            key = self.keys[self._idx]
            if self._is_available(key):
                return key
            self._idx = (self._idx + 1) % len(self.keys)
        return self.keys[self._idx]

    # ---- bookkeeping ----
    def record(self, key, tokens=0):
        with self._lock:
            now = time.time()
            self._usage[key]['min_req'].append(now)
            self._usage[key]['day_req'].append(now)
            if tokens > 0:
                self._usage[key]['min_tok'].append((now, tokens))
                self._usage[key]['day_tok'].append((now, tokens))

    def mark_exhausted(self, key):
        """Force-rotate away from a key that returned 429."""
        with self._lock:
            now = time.time()
            self._usage[key]['min_req'].extend([now] * self.BUFFERS['rpm'])
            self._idx = (self._idx + 1) % len(self.keys)

    def status(self):
        with self._lock:
            for i, key in enumerate(self.keys):
                self._clean(key)
                u = self._usage[key]
                rpm = len(u['min_req'])
                rpd = len(u['day_req'])
                tpm = sum(n for _, n in u['min_tok'])
                print(f"  Key {i+1:2d}: {rpm:3d}/{self.BUFFERS['rpm']} RPM  "
                      f"{rpd:4d}/{self.BUFFERS['rpd']} RPD  "
                      f"{tpm:6d}/{self.BUFFERS['tpm']} TPM")

key_manager = APIKeyManager(API_KEYS_CSV)

# Auto-scale MAX_WORKERS to match the number of loaded API keys
MAX_WORKERS = len(key_manager.keys)
print(f"Workers  : {MAX_WORKERS} (auto-scaled to match {MAX_WORKERS} API keys)")

## Step 4 — Load HotPotQA Data

In [ ]:
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
if N_SAMPLES is not None:
    random.seed(SEED)
    indices = random.sample(range(len(ds)), min(N_SAMPLES, len(ds)))
    samples = ds.select(indices)
else:
    samples = ds
print(f"Loaded {len(samples)} samples")

## Step 5 — Data Processing Utilities

In [ ]:
def process_context(context):
    """Flatten HotPotQA context into a list of candidate sentences."""
    candidates = []
    for title, sentences in zip(context['title'], context['sentences']):
        for i, sent in enumerate(sentences):
            candidates.append({'title': title, 'sent_id': i, 'text': sent})
    return candidates

def format_gold_supporting_facts(supporting_facts):
    return [{'title': t, 'sent_id': s}
            for t, s in zip(supporting_facts['title'], supporting_facts['sent_id'])]

## Step 6 — BM25 Retriever

In [ ]:
def bm25_retrieve(query, candidates, k=5):
    if not candidates:
        return []
    texts = ["passage: " + c['text'] for c in candidates]
    tokenized = [t.lower().split() for t in texts]
    bm25 = BM25Okapi(tokenized)
    scores = bm25.get_scores(query.lower().split())
    top_idx = scores.argsort()[-k:][::-1]
    return [dict(**candidates[i], score=float(scores[i])) for i in top_idx]

print("BM25 retriever ready.")

## Step 7 — LLM Client (Multi-Key Groq)

Each `generate()` call:
1. Asks the KeyManager for the best available key
2. Makes the API call
3. Records token usage back to the KeyManager
4. On 429 → marks key exhausted → rotates → retries (up to 3×)

In [ ]:
class GroqClient:
    def __init__(self, model_name, km):
        self.model_name = model_name
        self.km = km
        self._clients = {}
        self._clock = threading.Lock()

    def _client_for(self, api_key):
        with self._clock:
            if api_key not in self._clients:
                self._clients[api_key] = Groq(api_key=api_key)
            return self._clients[api_key]

    def generate(self, prompt, system_prompt=None, max_retries=3):
        for attempt in range(max_retries):
            api_key = self.km.get_key()
            client = self._client_for(api_key)
            try:
                msgs = []
                if system_prompt:
                    msgs.append({"role": "system", "content": system_prompt})
                msgs.append({"role": "user", "content": prompt})
                resp = client.chat.completions.create(
                    model=self.model_name, messages=msgs,
                    max_tokens=512, temperature=0.1,
                )
                tokens = resp.usage.total_tokens if resp.usage else 0
                self.km.record(api_key, tokens)
                return resp.choices[0].message.content
            except Exception as e:
                if '429' in str(e) or 'rate_limit' in str(e).lower():
                    self.km.mark_exhausted(api_key)
                    continue
                print(f"  LLM error: {e}")
                return ""
        print("  All retries exhausted")
        return ""

    @staticmethod
    def parse_json_output(text):
        try: return json.loads(text)
        except json.JSONDecodeError: pass
        m = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
        if m:
            try: return json.loads(m.group(1))
            except json.JSONDecodeError: pass
        m = re.search(r"(\{.*\})", text, re.DOTALL)
        if m:
            try: return json.loads(m.group(1))
            except json.JSONDecodeError: pass
        return {"answer": "JSON_PARSE_ERROR", "supporting_facts": [], "raw_output": text}

    def predict(self, prompt):
        return self.parse_json_output(self.generate(prompt))

llm = GroqClient(GROQ_MODEL, key_manager)
print("LLM client ready (multi-key).")

## Step 8 — Prompt Construction

In [ ]:
def construct_prompt(question, retrieved_sentences):
    context_str = ""
    for i, item in enumerate(retrieved_sentences, 1):
        context_str += (f"[{i}] Title: {item['title']}\n"
                        f"    Sentence ID: {item['sent_id']}\n"
                        f"    Text: {item['text']}\n\n")
    return f"""You are a helpful assistant for Question Answering.
Answer the following question based ONLY on the provided context sentences.
You must also identify which sentences support your answer.

Context:
{context_str}

Question: {question}

Instructions:
1. Provide a short, concise answer.
2. List the supporting facts as title + sent_id pairs.
3. Use EXACT titles and sent_ids from the context.
4. Output valid JSON only.

Format:
{{{{
  "answer": "...",
  "supporting_facts": [{{{{"title": "...", "sent_id": N}}}}, ...]
}}}}"""

## Step 9 — Evaluator

In [ ]:
def normalize_answer(s):
    s = str(s) if s is not None else ""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    return ' '.join(s.split())

def answer_f1(pred, gold):
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    common = collections.Counter(pt) & collections.Counter(gt)
    ns = sum(common.values())
    if ns == 0: return 0.0
    p, r = ns / len(pt), ns / len(gt)
    return (2 * p * r) / (p + r)

def answer_em(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def sp_metrics(pred_sp, gold_sp):
    def to_set(lst):
        return {(x['title'], x['sent_id']) if isinstance(x, dict) else (x[0], x[1]) for x in lst}
    ps, gs = to_set(pred_sp), to_set(gold_sp)
    tp = len(ps & gs)
    prec = tp / len(ps) if ps else 0.0
    rec  = tp / len(gs) if gs else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    em   = 1.0 if ps == gs and len(gs) > 0 else 0.0
    return {'sp_em': em, 'sp_f1': f1, 'sp_prec': prec, 'sp_recall': rec}

## Step 10 — Run Vanilla RAG (Parallel Execution)

14 workers process samples concurrently. Each worker:
1. Retrieves via BM25
2. Calls the LLM (KeyManager auto-selects a fresh key)
3. Returns the result

In [ ]:
def process_sample(sample):
    """Process a single sample (thread-safe)."""
    t0 = time.time()
    question   = sample['question']
    candidates = process_context(sample['context'])
    retrieved  = bm25_retrieve(question, candidates, k=TOP_K)
    prompt     = construct_prompt(question, retrieved)
    resp       = llm.predict(prompt)
    elapsed    = time.time() - t0
    gold_sp    = format_gold_supporting_facts(sample['supporting_facts'])
    return {
        'pred': {'answer': resp.get('answer', ''), 'supporting_facts': resp.get('supporting_facts', [])},
        'gold': {'answer': sample['answer'], 'supporting_facts': gold_sp},
        'detail': {
            'id': sample['id'], 'question': question,
            'gold_answer': sample['answer'], 'gold_sp': gold_sp,
            'pred_answer': resp.get('answer', ''), 'pred_sp': resp.get('supporting_facts', []),
            'retrieved_context': retrieved, 'time_taken': elapsed,
            'pipeline': PIPELINE_NAME, 'raw_prediction': resp,
        }
    }

# --- Parallel execution ---
experiment_start = datetime.now()
print(f"Starting {PIPELINE_NAME} at {experiment_start.strftime('%H:%M:%S')}")
print(f"{len(samples)} samples × {MAX_WORKERS} workers\n")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_map = {executor.submit(process_sample, s): i for i, s in enumerate(samples)}
    for future in tqdm(as_completed(future_map), total=len(future_map), desc=PIPELINE_NAME):
        try:
            r = future.result()
            r['idx'] = future_map[future]
            results.append(r)
        except Exception as e:
            print(f"  Error on sample {future_map[future]}: {e}")

results.sort(key=lambda x: x['idx'])
predictions = [r['pred']   for r in results]
golds       = [r['gold']   for r in results]
details     = [r['detail'] for r in results]

experiment_end = datetime.now()
total_time = (experiment_end - experiment_start).total_seconds()
print(f"\nDone in {total_time:.1f}s ({total_time/len(samples):.2f}s/sample)")

## Step 11 — Evaluation

In [ ]:
metrics = {'em':0,'f1':0,'sp_em':0,'sp_f1':0,'sp_prec':0,'sp_recall':0,'joint_em':0,'joint_f1':0}
for pred, gold in zip(predictions, golds):
    em = answer_em(pred['answer'], gold['answer'])
    f1 = answer_f1(pred['answer'], gold['answer'])
    sp = sp_metrics(pred['supporting_facts'], gold['supporting_facts'])
    metrics['em'] += em;  metrics['f1'] += f1
    metrics['sp_em'] += sp['sp_em'];  metrics['sp_f1'] += sp['sp_f1']
    metrics['sp_prec'] += sp['sp_prec'];  metrics['sp_recall'] += sp['sp_recall']
    metrics['joint_em'] += em * sp['sp_em'];  metrics['joint_f1'] += f1 * sp['sp_f1']

n = len(predictions)
for k in metrics: metrics[k] /= n

print(f"{'='*50}")
print(f"  {PIPELINE_NAME} Results ({n} samples)")
print(f"{'='*50}")
for k, v in metrics.items():
    print(f"  {k:20s}: {v:.4f}")
print(f"{'='*50}")
print(f"  Total: {total_time:.1f}s | Avg: {total_time/n:.2f}s/sample")

## RAGAS Faithfulness Evaluation

Measures what fraction of the generated answer's claims can be inferred from the retrieved context.

In [ ]:
import sys
import time
import copy
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---------------------------------------------------------------------
# Hotfix: RAGAS can reference langchain_community.chat_models.vertexai
# ---------------------------------------------------------------------
try:
    import langchain_google_vertexai
    import langchain_community

    if not hasattr(langchain_community, "chat_models"):
        class Dummy:
            pass

        langchain_community.chat_models = Dummy()

    sys.modules["langchain_community.chat_models.vertexai"] = langchain_google_vertexai

except ImportError:
    pass


print("\nRunning RAGAS Faithfulness evaluation: 1 key per worker per wave...")

warnings.filterwarnings("ignore", category=DeprecationWarning, module="ragas")

from ragas import evaluate as ragas_evaluate
from ragas.metrics import faithfulness
from ragas.run_config import RunConfig
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset


# ---------------------------------------------------------------------
# Tunables
# ---------------------------------------------------------------------
RAGAS_MODEL = "llama-3.3-70b-versatile"

WAVE_COOLDOWN = 15       # seconds between waves
SAMPLE_TIMEOUT = 240     # per-sample RAGAS timeout
MAX_RETRIES = 3          # retries per sample
BASE_MAX_TOKENS = 4096   # first attempt output budget
RETRY_MAX_TOKENS = 8192  # retry output budget if generation is cut off

ragas_run_cfg = RunConfig(
    timeout=SAMPLE_TIMEOUT,
    max_retries=3,
    max_wait=30,
)


# ---------------------------------------------------------------------
# Validate required notebook variables
# ---------------------------------------------------------------------
required_vars = ["details", "metrics", "key_manager"]
missing = [v for v in required_vars if v not in globals()]

if missing:
    raise NameError(
        f"Missing required notebook variable(s): {missing}. "
        "Make sure details, metrics, and key_manager are created before this cell."
    )


# ---------------------------------------------------------------------
# Build per-sample dicts once
# ---------------------------------------------------------------------
sample_dicts = []

for d in details:
    ctx = [
        item["text"]
        for item in d.get("retrieved_context", [])
        if isinstance(item, dict) and "text" in item
    ]

    sample_dicts.append(
        {
            "question": d.get("question", ""),
            "answer": d.get("pred_answer", ""),
            "contexts": ctx if ctx else [""],
        }
    )


n_keys = len(key_manager.keys)
n_samples = len(sample_dicts)
MAX_WORKERS = n_keys

if n_keys == 0:
    raise ValueError("No API keys found in key_manager.keys")

print(
    f"{n_samples} samples | {n_keys} keys | "
    f"{MAX_WORKERS} workers | ~{(n_samples + n_keys - 1) // n_keys} waves"
)


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def build_dataset_for_sample(sample_idx):
    sample = sample_dicts[sample_idx]

    return Dataset.from_dict(
        {
            "question": [sample["question"]],
            "answer": [sample["answer"]],
            "contexts": [sample["contexts"]],
        }
    )


def build_metric_for_key(api_key, max_tokens):
    """
    Make a private faithfulness metric per worker.

    Do NOT do this in parallel:
        faithfulness.llm = evaluator_llm

    That mutates the global faithfulness object and lets threads overwrite
    each other's key-specific LLM.

    Do NOT use deepcopy either:
        copy.deepcopy(faithfulness)

    That can fail with:
        cannot pickle '_thread.RLock' object

    Shallow copy is the safer legacy-RAGAS fix.
    """
    llm = ChatGroq(
        model=RAGAS_MODEL,
        groq_api_key=api_key,
        temperature=0,
        max_tokens=max_tokens,
        timeout=SAMPLE_TIMEOUT,
    )

    evaluator_llm = LangchainLLMWrapper(llm)

    metric = copy.copy(faithfulness)
    metric.llm = evaluator_llm

    return metric


def is_nan_like(x):
    return x is None or x != x


def evaluate_one(sample_idx, api_key, key_no):
    """
    Evaluate one sample using one specific Groq API key.

    Returns:
        sample_idx, score, final_key_no
    """
    ds = build_dataset_for_sample(sample_idx)

    current_key = api_key
    current_key_no = key_no

    for attempt in range(MAX_RETRIES):
        try:
            max_tokens = BASE_MAX_TOKENS if attempt == 0 else RETRY_MAX_TOKENS
            metric = build_metric_for_key(current_key, max_tokens=max_tokens)

            # Track this request against the exact key being used.
            key_manager.record(current_key)

            res = ragas_evaluate(
                ds,
                metrics=[metric],
                run_config=ragas_run_cfg,
            )

            score = res.to_pandas()["faithfulness"].iloc[0]

            return sample_idx, score, current_key_no

        except Exception as e:
            err = str(e)
            err_lower = err.lower()

            is_rate_limit = (
                "429" in err
                or "rate_limit" in err_lower
                or "rate limit" in err_lower
                or "too many requests" in err_lower
            )

            is_generation_cutoff = (
                "llmdidnotfinishexception" in err_lower
                or "generation was not completed" in err_lower
                or "increase the max_tokens" in err_lower
                or ("finish_reason" in err_lower and "length" in err_lower)
            )

            if is_rate_limit:
                key_manager.mark_exhausted(current_key)

                current_key = key_manager.get_key()
                current_key_no = key_manager.keys.index(current_key) + 1

                wait = 5 * (attempt + 1)

                print(
                    f"    [sample {sample_idx + 1}] 429/rate-limit → "
                    f"rotated to key #{current_key_no}, "
                    f"retry {attempt + 1}/{MAX_RETRIES} after {wait}s"
                )

                time.sleep(wait)

            elif is_generation_cutoff and attempt < MAX_RETRIES - 1:
                wait = 3 * (attempt + 1)

                print(
                    f"    [sample {sample_idx + 1}] generation cut off → "
                    f"retry {attempt + 1}/{MAX_RETRIES} "
                    f"with max_tokens={RETRY_MAX_TOKENS} after {wait}s"
                )

                time.sleep(wait)

            else:
                print(f"    [sample {sample_idx + 1}] error: {e}")
                return sample_idx, None, current_key_no

    return sample_idx, None, current_key_no


# ---------------------------------------------------------------------
# Run in waves of n_keys
# ---------------------------------------------------------------------
scores = [None] * n_samples
wave = 0
i = 0

while i < n_samples:
    wave += 1

    wave_end = min(i + n_keys, n_samples)
    wave_size = wave_end - i

    print(f"\n  Wave {wave}: samples {i + 1}-{wave_end} ({wave_size} in parallel)")
    print(f"  Using keys: 1-{wave_size}")

    futures = {}

    with ThreadPoolExecutor(max_workers=wave_size) as pool:
        for j in range(wave_size):
            sample_idx = i + j

            # Exact mapping per wave:
            # sample 1  -> key 1
            # sample 2  -> key 2
            # ...
            # sample 31 -> key 31
            #
            # Next wave repeats key 1-31 after cooldown.
            key_idx = j
            api_key = key_manager.keys[key_idx]
            key_no = key_idx + 1

            fut = pool.submit(evaluate_one, sample_idx, api_key, key_no)
            futures[fut] = (sample_idx, key_no)

        for fut in as_completed(futures):
            scheduled_sample_idx, scheduled_key_no = futures[fut]

            try:
                idx, score, final_key_no = fut.result()

            except Exception as e:
                idx = scheduled_sample_idx
                score = None
                final_key_no = scheduled_key_no
                print(f"    sample {idx + 1:2d}  key #{final_key_no:2d}  → FAIL | {e}")

            tag = f"{score:.4f}" if not is_nan_like(score) else "FAIL"

            print(f"    sample {idx + 1:2d}  key #{final_key_no:2d}  → {tag}")

            scores[idx] = score


    i = wave_end

    if i < n_samples:
        print(f"  Cooling down {WAVE_COOLDOWN}s before next wave...")
        time.sleep(WAVE_COOLDOWN)


# ---------------------------------------------------------------------
# Aggregate
# ---------------------------------------------------------------------
valid = [s for s in scores if not is_nan_like(s)]

avg_faithfulness = sum(valid) / len(valid) if valid else 0.0

metrics["faithfulness"] = avg_faithfulness
faithfulness_scores = scores

print(f"\n{'=' * 50}")
print(f"  RAGAS Faithfulness: {avg_faithfulness:.4f}")
print(f"  (computed on {len(valid)}/{n_samples} samples)")
print(f"{'=' * 50}")

key_manager.status()


## Step 12 — Save Results

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
ts = experiment_start.strftime('%Y%m%d_%H%M%S')
output = {
    'args': {'pipeline': PIPELINE_NAME, 'model': GROQ_MODEL, 'top_k': TOP_K,
             'n_samples': N_SAMPLES, 'max_workers': MAX_WORKERS},
    'metrics': metrics,
    'timing': {'start': experiment_start.isoformat(), 'end': experiment_end.isoformat(),
               'total_s': total_time, 'avg_s': total_time / n},
    'details': details,
    'faithfulness_per_sample': globals().get('faithfulness_scores', []),
}
out_file = f'{RESULTS_DIR}/vanilla_{ts}_results.json'
with open(out_file, 'w') as f:
    json.dump(output, f, indent=2)
print(f"Saved → {out_file}")

## Step — Export Metrics CSV


In [ ]:
# --- Export metrics to CSV ---------------------------------------------------
import csv, os

CSV_PATH = f"{BASE_DIR}/Results/BestEmbeddingComparison/CSVOutputs/BestEmbeddingComparison_results.csv"
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

# Derive a unique pipeline label  (e.g. "VanillaRAG_FAISS", "GraphRAG_Qdrant")
_store = RESULTS_DIR.rstrip("/").split("/")[-1]            # faiss / lancedb / qdrant
_label = PIPELINE_NAME
if _store.lower() not in _label.lower():                   # VanillaRAG has no suffix
    _label = f"{PIPELINE_NAME}_{_store.upper()}"

row = {
    "Pipeline":           _label,
    "em":                 round(metrics.get("em", 0), 4),
    "f1":                 round(metrics.get("f1", 0), 4),
    "sp_em":              round(metrics.get("sp_em", 0), 4),
    "sp_f1":              round(metrics.get("sp_f1", 0), 4),
    "sp_prec":            round(metrics.get("sp_prec", 0), 4),
    "sp_recall":          round(metrics.get("sp_recall", 0), 4),
    "joint_em":           round(metrics.get("joint_em", 0), 4),
    "joint_f1":           round(metrics.get("joint_f1", 0), 4),
    "RAGAS Faithfulness": round(metrics.get("faithfulness", 0), 4),
}

header = list(row.keys())
write_header = not os.path.exists(CSV_PATH)

with open(CSV_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=header)
    if write_header:
        w.writeheader()
    w.writerow(row)

print(f"Appended metrics row to {CSV_PATH}")
print("  ", row)

## Step 13 — Inspect Predictions & Key Usage

In [ ]:
for d in details[:5]:
    print(f"Q: {d['question']}")
    print(f"  Gold: {d['gold_answer']}")
    print(f"  Pred: {d['pred_answer']}")
    print(f"  Retrieved: {[(r['title'], r['sent_id']) for r in d['retrieved_context']]}")
    print()

print("\n--- API Key Usage ---")
key_manager.status()